In [1]:
import os
import pandas as pd
from ftplib import FTP
import os
from pathlib import Path
import requests
from ftplib import FTP
from asf_search import ASFSession, ASFSearchResults, granule_search
import requests
from tqdm import tqdm
from pathlib import Path

# read CSV pairs

In [2]:
# === 配置部分 ===
csv_path = r"E:\NWP\CS2_S1_matched\time_match_2023.csv"
sentinel1_dir = r"E:\NWP\CS2_S1_matched\Sentinel-1"
cryosat2_dir = r"E:\NWP\CS2_S1_matched\CS2\2023"


In [3]:
# === 读取CSV ===
df = pd.read_csv(csv_path)
sceneName = df['sceneName'].tolist() 
url_list = df['s1_url']  # 假设CSV中有一个列名为'url'，存储了下载链接

# 提取 'url' 列为列表（确保去掉缺失值）
urls = df['s1_url'].dropna().tolist()


这里的SIN_2__后面有两个下划线不可思议

In [4]:
# === CryoSat-2 文件路径构造 ===
def generate_cs2_L1_name(cs2_filename):
    return cs2_filename.replace("SIN_2__", "SIN_1B_")

# === 遍历下载 ===
cs2_l1_name_list = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    cs2_original_name = os.path.basename(row['cs2_path'])
    cs2_l1_name = generate_cs2_L1_name(cs2_original_name)
    cs2_l1_name_list.append(cs2_l1_name)

print(f"Downloading Sentinel-1: {sceneName}")
print(f"Downloading CryoSat-2: {cs2_l1_name}")
print(len(cs2_l1_name))

100%|██████████| 360/360 [00:00<00:00, 37850.93it/s]

58


In [5]:
print(cs2_l1_name_list)

['CS_OFFL_SIR_SIN_1B_20230329T173455_20230329T173845_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230105T194558_20230105T194809_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230105T180803_20230105T180905_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230630T102149_20230630T102404_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230629T124858_20230629T125249_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230629T111216_20230629T111426_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230628T133846_20230628T134259_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230628T115922_20230628T120435_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230628T102400_20230628T102559_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230628T102400_20230628T102559_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230627T111042_20230627T111348_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230626T134124_20230626T134456_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230626T134124_20230626T134456_E001.nc', 'CS_OFFL_SIR_SIN_1B_20230626T120120_20230626T120632_E001.nc', 'CS_OFF

# download

## test

In [39]:
import asf_search as asf
name = sceneName[0]  # 假设你只需要第一个scene name进行搜索
results = asf.granule_search(name)
print(results)


{
  "features": [
    {
      "geometry": {
        "coordinates": [
          [
            [
              -89.305229,
              76.937294
            ],
            [
              -78.152473,
              80.1772
            ],
            [
              -99.358772,
              82.0299
            ],
            [
              -105.586288,
              78.316002
            ],
            [
              -89.305229,
              76.937294
            ]
          ]
        ],
        "type": "Polygon"
      },
      "properties": {
        "beamModeType": "EW",
        "browse": [
          "https://datapool.asf.alaska.edu/BROWSE/SA/S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83.jpg"
        ],
        "bytes": 59061,
        "centerLat": 79.5309,
        "centerLon": -93.0747,
        "fileID": "S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_049216_05EB02_5C83-METADATA_GRD_MD",
        "fileName": "S1A_EW_GRDM_1SDH_20230630T134749_20230630T134853_0

In [ ]:
import pandas as pd
import asf_search as asf

url = url_list[0:2]  # 假设你只需要第一个URL进行下载
# result = asf.granule_search(name)
# print(f"搜索结果: {result}")
asf.download_urls(urls=url, path =  sentinel1_dir)

ChunkedEncodingError: ('Connection broken: IncompleteRead(15656453 bytes read, 255839782 more expected)', IncompleteRead(15656453 bytes read, 255839782 more expected))

## directly download 
## the token is not finished


In [11]:
import os
import requests
from tqdm import tqdm
from time import sleep
from requests.auth import HTTPBasicAuth

# ✅ 你的 ASF 账号信息
ASF_USERNAME = "muyee5229"
ASF_PASSWORD = "syJ3958101701!"

def download_with_auth(url, output_path, max_retries=5):
    if isinstance(url, (list, tuple, pd.Series)):
        # 处理多个URL的情况
        for single_url in url:
            download_with_auth(single_url, output_path, max_retries)
        return
    filename = url.split('/')[-1]  # 从URL中提取文件名
    save_path = os.path.join(output_path, filename)
    if os.path.exists(save_path):
        print(f"✅ 已存在: {save_path}")
        return

    for attempt in range(max_retries):
        try:
            with requests.get(url, stream=True, timeout=60, auth=HTTPBasicAuth(ASF_USERNAME, ASF_PASSWORD)) as r:
                r.raise_for_status()
                total_size = int(r.headers.get('content-length', 0))
                with open(save_path, 'wb') as f, tqdm(
                    total=total_size,
                    unit='B', unit_scale=True,
                    desc=save_path
                ) as bar:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            bar.update(len(chunk))
            print(f"✅ 下载完成: {save_path}")
            return
        except Exception as e:
            print(f"⚠️ 第 {attempt+1} 次下载失败: {e}")
            sleep(5)
    print(f"❌ 超过最大重试次数失败: {url}")


## try script on github

In [8]:
# === CryoSat-2 FTP 设置 ===
ftp_host = "science-pds.cryosat.esa.int"
ftp_user = "cryosat507"
ftp_pass = "qOnliGbm"
# 连接FTP
ftp = FTP(ftp_host)
ftp.login(ftp_user, ftp_pass)



'230 Login successful.'

In [65]:
from ftplib import FTP
from pathlib import Path
import re
import time

def extract_year_month(filename):
    # 适配 1B 数据的时间格式
    match = re.search(r'_(\d{4})(\d{2})(\d{2})T', filename)
    if match:
        return match.group(1), match.group(2)

    raise ValueError("❌ Filename does not match known CryoSat-2 format.")

def download_cryosat2(filename: str, output_dir: Path, ftp_user='your_username', ftp_pass='your_password'):
    """
    Downloads a CryoSat-2 file (e.g., SIR_SIN_1B) from ESA FTP, based on its name.
    """

    year, month = extract_year_month(filename)
    remote_path = f'/SIR_SIN_L1/{year}/{month}/'   # ← 确认你的路径是这个结构

    output_dir.mkdir(parents=True, exist_ok=True)
    local_file_path = output_dir / filename

    with FTP('science-pds.cryosat.esa.int') as ftp:
        ftp.login(ftp_user, ftp_pass)
        ftp.cwd(remote_path)

        files = ftp.nlst()
        if filename not in files:
            raise FileNotFoundError(f"{filename} not found in {remote_path}")

        with open(local_file_path, 'wb') as f:
            ftp.retrbinary(f"RETR {filename}", f.write)

    print(f"✅ Downloaded: {filename} to {local_file_path}")

def download_cryosat2_batch(file_list, output_dir, ftp_user, ftp_pass):
    with FTP('science-pds.cryosat.esa.int') as ftp:
        ftp.login(ftp_user, ftp_pass)
        
        for filename in file_list:
            # Extract year/month
            match = re.match(r'CS_OFFL_SIR_SIN_1B_(\d{4})(\d{2})\d{2}T', filename)
            if not match:
                print(f"Skipping {filename} (bad format)")
                continue
            year, month = match.group(1), match.group(2)
            remote_path = f"/SIR_SIN_L1/{year}/{month}"
            local_path = output_dir / filename
            output_dir.mkdir(parents=True, exist_ok=True)
            
            try:
                ftp.cwd(remote_path)
                files = ftp.nlst()
                if filename not in files:
                    print(f"{filename} not found in {remote_path}")
                    continue
                with open(local_path, 'wb') as f:
                    ftp.retrbinary(f"RETR {filename}", f.write)
                print(f"Downloaded: {filename}")
                time.sleep(1.5)  # polite delay
            except Exception as e:
                print(f"Failed to download {filename}: {e}")

In [58]:
from pathlib import Path

temp = cs2_l1_name_list[1:4]  # 假设这个是 'CS_OFFL_SIR_SIN_L1_20230101T000000_20230101T235959_001.nc'
cryosat2_dir = Path("./cryosat2_downloads")
for i in temp:
    download_cryosat2(i, cryosat2_dir, ftp_user= ftp_user, ftp_pass=ftp_pass)


✅ Downloaded: CS_OFFL_SIR_SIN_1B_20230105T194558_20230105T194809_E001.nc to cryosat2_downloads\CS_OFFL_SIR_SIN_1B_20230105T194558_20230105T194809_E001.nc
✅ Downloaded: CS_OFFL_SIR_SIN_1B_20230105T180803_20230105T180905_E001.nc to cryosat2_downloads\CS_OFFL_SIR_SIN_1B_20230105T180803_20230105T180905_E001.nc
✅ Downloaded: CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc to cryosat2_downloads\CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc


In [ ]:
from pathlib import Path

download_cryosat2_batch(cs2_l1_name_list, Path(cryosat2_dir), ftp_user, ftp_pass)


Downloaded: CS_OFFL_SIR_SIN_1B_20230329T173455_20230329T173845_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230105T194558_20230105T194809_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230105T180803_20230105T180905_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230630T102149_20230630T102404_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230629T124858_20230629T125249_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230629T111216_20230629T111426_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230628T133846_20230628T134259_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230628T115922_20230628T120435_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230628T102400_20230628T102559_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230628T102400_20230628T102559_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230627T111042_20230627T111348_E001.nc
Downloaded: CS_OFFL_SIR_SIN_1B_20230626T134124_20230626T134456_E001.nc
Downlo

In [6]:
from ftplib import FTP, error_perm
import re
import time
from pathlib import Path

def download_cryosat2_batch_resumable(file_list, output_dir, ftp_user, ftp_pass, retry_limit=3, delay=2):
    """
    批量下载 CryoSat-2 数据，具备断点续传和容错机制。
    
    参数:
        file_list: 所有需要下载的文件名列表
        output_dir: 下载保存路径 (Path)
        ftp_user: FTP 用户名
        ftp_pass: FTP 密码
        retry_limit: 每个文件最多重试次数
        delay: 每次下载后的延迟秒数
    """
    
    def connect_ftp():
        ftp = FTP('science-pds.cryosat.esa.int', timeout=30)
        ftp.login(ftp_user, ftp_pass)
        return ftp

    # 确保目标路径存在
    output_dir.mkdir(parents=True, exist_ok=True)

    ftp = None
    for filename in file_list:
        # 检查本地文件是否已存在
        local_file_path = output_dir / filename
        if local_file_path.exists():
            print(f"✅ 已存在，跳过: {filename}")
            continue

        # 提取年份和月份
        match = re.match(r'CS_OFFL_SIR_SIN_1B_(\d{4})(\d{2})\d{2}T', filename)
        if not match:
            print(f"⚠️ 格式不匹配，跳过: {filename}")
            continue
        year, month = match.group(1), match.group(2)
        remote_path = f"/SIR_SIN_L1/{year}/{month}"

        attempt = 0
        while attempt < retry_limit:
            try:
                if ftp is None:
                    ftp = connect_ftp()

                ftp.cwd(remote_path)
                files = ftp.nlst()
                if filename not in files:
                    print(f"❌ 文件未找到: {filename} in {remote_path}")
                    break

                with open(local_file_path, 'wb') as f:
                    ftp.retrbinary(f"RETR {filename}", f.write)
                print(f"✅ 成功下载: {filename}")
                time.sleep(delay)
                break  # 成功后退出重试循环

            except (OSError, EOFError, error_perm) as e:
                print(f"⚠️ 下载失败（第 {attempt + 1} 次）: {filename} -> {e}")
                attempt += 1
                time.sleep(5)
                # 尝试重新连接 FTP
                try:
                    if ftp:
                        ftp.quit()
                except:
                    pass
                ftp = None  # 触发重连
        else:
            print(f"❌ 放弃下载: {filename} after {retry_limit} attempts")

    # 最后关闭 FTP 连接
    if ftp:
        ftp.quit()


In [9]:
from pathlib import Path

download_cryosat2_batch_resumable(cs2_l1_name_list, Path(cryosat2_dir), ftp_user, ftp_pass)

✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230329T173455_20230329T173845_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230105T194558_20230105T194809_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230105T180803_20230105T180905_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230630T115732_20230630T120238_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230630T102149_20230630T102404_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230629T124858_20230629T125249_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230629T111216_20230629T111426_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230628T133846_20230628T134259_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230628T115922_20230628T120435_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230628T102400_20230628T102559_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230628T102400_20230628T102559_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230627T111042_20230627T111348_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230626T134124_20230626T134456_E001.nc
✅ 已存在，跳过: CS_OFFL_SIR_SIN_1B_20230

KeyboardInterrupt: 